# Week 1 — Research Question
**Track:** FlyRank ML Internship — Applied Search Intelligence
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring


## 1. My lane, and why

I'm picking **Lane 2: Refresh / Content Opportunity Scoring** (a predefined lane, not freestyle).

Reasons:

- It's the lane the starter pipeline (`scripts/01`–`05`) already builds end-to-end, and the starter dataset
  (`data/raw/content_refresh_anonymized.csv`) was purpose-built to support it — I don't need the gated
  warehouse release to get a real first result this week.
- The output is directly actionable: a **ranked review queue**, not just a report. Someone with limited
  review capacity can open the top of the list and start working immediately.
- The lane guide's own benchmark numbers (Precision@50 going from 0.240 with the hand-rule baseline to
  0.740 with a random forest) show there's real signal to learn here, on this exact dataset — that's a
  strong sign 7 weeks of work will find something, not nothing.
- It naturally extends toward the stronger, future-looking version the guide recommends
  (`features from prior 90 days -> decline or recovery over next 30 days`) once I have warehouse access
  in Weeks 3+, so I'm not boxing myself into a beginner proxy label forever.

I'm keeping this provisional, as the assignment allows — I can confirm or change it by end of Week 4.


## 2. The question: decision, action, cost of a wrong call

**Search question:**
Given a client's existing content inventory and 90 days of observed search/engagement signals, which
pages should a content reviewer look at *first* when they only have time to review a handful this week?

**Unit of analysis:** one **content item** (one row = one page, identified by `content_id`), evaluated
per client. This matches the grain of the starter dataset and of `dim_content` in the warehouse.

**The decision this improves:** right now, a reviewer with limited time has to pick which pages to check
for a possible refresh, expansion, or protection action, out of a client's whole inventory (thousands of
pages per client in the warehouse). Today that pick is either arbitrary or based on the product's own
rule-based `health_score`/`priority_score` (which I'm not allowed to reuse as a feature or label — see
Section 4). My job is to build an independent, evidence-based ranking of the same decision.

**The action someone takes:** a human reviewer opens the top of my ranked queue and, for each page,
sees a **reason code** (e.g. "declining with demand", "page-one decay risk", "low CTR at a good position")
and a suggested action — refresh the content, rewrite title/meta, expand thin content, or simply monitor.
They do NOT get an automated edit; they get a shorter, better-ordered to-do list.

**The cost of a wrong recommendation:**

- *False positive* (I flag a page as worth reviewing, but it wasn't really at risk or opportunity):
  wastes a reviewer's time — low-severity cost, but it erodes trust in the tool if it happens often.
- *False negative* (a page that's genuinely declining or under-performing never makes the list):
  higher-severity cost — a page that's losing visibility keeps losing it, unnoticed, until the next
  full audit. For a client paying for content strategy, that's lost traffic and, eventually, lost
  revenue that nobody caught in time.
- Because false negatives are costlier than false positives here, I plan to care more about **recall
  within a fixed review budget** (Precision@K / a top-K review) than about raw accuracy — this matches
  how the lane guide frames validation for ranked-queue lanes.

**Why data or ML can help at all (and not just a fixed rule):**
The starter pipeline already shows a hand-written rule (`baseline_refresh_score`) scoring
Precision@50 = 0.240, while a random forest trained on the same observable signals reaches
Precision@50 = 0.740 on a client-holdout split. That's a ~3x improvement in "how many of the top 50
flagged pages actually matched the label" — evidence that the relationship between the input signals
(impressions, position, CTR, freshness, engagement, word count, etc.) and "this page needs review" is
more complex than a linear hand-tuned weighting can capture, and that a model can learn interactions a
fixed rule misses. This is not "train a model because ML is trendy" — it's "a fixed rule already exists
and a model has been shown, on this data, to beat it materially."


## 3. Quick look at the data (real numbers from the starter dataset)

Loading `data/raw/content_refresh_anonymized.csv` directly (the same file the starter pipeline uses)
to check that this lane is actually worth pursuing before committing 7 weeks to it.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique()}")
print(f"Columns: {len(df.columns)}")
df.head(3)


Rows: 30,000
Unique clients: 32
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


**Number 1 — there's a real, sizeable "declining" population to prioritize among.**

`trend_direction` is a precalculated bucket I'll treat as *context*, not a label I copy directly (per
the lane guide's warning about circular results) — but it's a useful sanity check that the phenomenon
I want to rank exists in volume, not as a rare edge case.


In [2]:
trend_counts = df['trend_direction'].value_counts()
print(trend_counts)
print(f"\nShare of pages currently trending down: {(df['trend_direction']=='down').mean():.1%}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Share of pages currently trending down: 54.2%


**Number 2 — a meaningful chunk of pages get real search exposure but zero clicks.**

This is exactly the kind of "visible but under-performing" page a refresh/CTR reviewer would want
surfaced — and it's large enough to be a real prioritization problem, not noise.


In [3]:
has_impr_zero_clicks = (df['impressions_90d'] > 0) & (df['clicks_90d'] == 0)
print(f"Pages with impressions_90d > 0 but clicks_90d == 0: {has_impr_zero_clicks.sum():,} "
      f"({has_impr_zero_clicks.mean():.1%} of all pages)")
print(f"Median impressions_90d across all pages: {df['impressions_90d'].median():.0f}")
print(f"Median clicks_90d across all pages: {df['clicks_90d'].median():.0f}")


Pages with impressions_90d > 0 but clicks_90d == 0: 13,204 (44.0% of all pages)
Median impressions_90d across all pages: 731
Median clicks_90d across all pages: 1


**Number 3 — the starter model already demonstrates the lane is learnable, not just describable.**

These numbers come straight from the repo's own committed `outputs/model_report.md` /
`outputs/model_results.json` (produced by `python scripts/run_all.py` on this same CSV), not from my
imagination — I'm quoting them here as the evidence for picking this lane, and I'll re-derive them
myself in Week 2 rather than take them on faith.


In [4]:
# Baseline vs. trained model, as reported by the starter pipeline on this dataset
# (source: outputs/model_report.md, outputs/model_results.json)
results = pd.DataFrame({
    "method": ["baseline rules", "logistic regression", "decision tree", "random forest"],
    "roc_auc": [0.627, 0.700, 0.742, 0.750],
    "average_precision": [0.468, 0.522, 0.575, 0.618],
    "precision_at_50": [0.240, 0.400, 0.540, 0.740],
})
results


,method,roc_auc,average_precision,precision_at_50
0,baseline rules,0.627,0.468,0.24
1,logistic regression,0.700,0.522,0.40
2,decision tree,0.742,0.575,0.54
3,random forest,0.750,0.618,0.74


Reading that last row plainly: of the top 50 pages the random forest ranks first, about **37** turn
out to match the (proxy) label, versus about **12** for the fixed hand-rule. That's the concrete
evidence — from this data, not a general claim about ML — that a learned ranking is worth building here.


## 4. Careful words: what I can and can't claim

**I can claim:**
- Which *observed* signals (impressions, clicks, position, CTR, freshness, word count, engagement,
  scroll rate) are associated with a page currently being labeled "declining" in this dataset.
- That a learned model, evaluated with client-holdout validation, ranks candidate pages for review
  better than a fixed hand-written rule *on this data and this proxy label*.
- That my ranked queue is a **decision-support** tool: it orders limited review capacity toward the
  pages most likely to be worth a human's time.

**I cannot claim:**
- That I know why any specific page is losing visibility in Google's actual ranking algorithm — I have
  no access to Google's ranking logic, only observed outcomes.
- That refreshing a flagged page will *cause* it to recover — that requires a controlled experiment
  (e.g. before/after with a control group), which this dataset alone cannot give me.
- That `trend_direction == "down"` (the starter label) is the *true* definition of "declining" — it's a
  proxy calculated from the current window, not a future outcome, and I plan to move toward a
  future-window label (`prior 90 days -> next 30 days`) once I have warehouse access.
- That my model works at the scale of the full ~79M-row warehouse — the numbers above are from a
  30,000-row anonymized starter slice for one narrow snapshot in time.
- Anything about AI search rankings, AI citations, or Google algorithm factors — those aren't measurable
  in this data at all.

I'll also avoid feeding any product-computed score (`health_score`, `priority_score`, `action_type`)
into my model as a feature, since the data was deliberately built without them — using one would make my
"discovery" circular (the model would just be re-learning FlyRank's existing rule).


## 5. Self-check

- [x] Picked one of the four predefined lanes (Lane 2 — Refresh / Content Opportunity Scoring) and
      explained why, backed by real numbers.
- [x] Named the decision being improved (which pages a reviewer checks first) and the action taken
      (open the ranked queue, read the reason code, act or monitor).
- [x] Stated the cost of a wrong call in both directions, and why false negatives worry me more here.
- [x] Showed 3 real numbers pulled from the starter CSV in executed code cells: the trend-direction
      distribution, the "impressions but zero clicks" population, and the baseline-vs-model
      Precision@50 comparison from the pipeline's own committed output.
- [x] Explained why this is not just "train a model" — a transparent baseline already exists, and the
      value is in showing (and later re-deriving myself) that a learned ranking materially beats it,
      under an honest validation split.
- [x] Used careful, hedged language throughout (observed/directional/decision-support), and explicitly
      listed what I cannot claim from this data.
- [ ] Not yet done, on purpose: locking in the final label definition and gaining warehouse access —
      that's Week 2+ work. This lane choice stays provisional until end of Week 4, as the assignment
      allows.
